<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/Analise-de-dados-Airbnb-Rio/blob/main/notebooks/analise_precos_airbnb_rio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quanto custa uma noite no Rio? Análise de preços do Airbnb

Análise dos anúncios do Airbnb no Rio de Janeiro com Python e pandas: limpeza dos dados, tratamento de outliers e respostas para 6 perguntas.

**Fonte:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Rio de Janeiro (CC BY 4.0). Data da extração: `AAAA-MM-DD`.

## Perguntas
1. Quais são os bairros mais caros e os mais baratos?
2. Quanto custa cada tipo de acomodação nos bairros mais procurados?
3. Superhosts cobram mais e são mais bem avaliados?
4. Quais são as comodidades mais comuns? Ter piscina deixa a diária mais cara?
5. Quantos anfitriões entraram no Airbnb a cada ano?
6. Quanto custa a diária média em dólar?

## Sumário
0. Configuração
1. Coleta dos dados
2. Entendimento inicial
3. Tratamento
4. Outliers
5. Transformações
6. Análises
7. Exportação
8. Conclusões

## 0. Configuração

In [ ]:
# Importa as bibliotecas usadas no projeto.
# Por quê: pandas/numpy para tratar os dados, matplotlib/seaborn para os gráficos, requests para a API do Banco Central.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Mostra todas as colunas ao exibir um DataFrame (a base tem mais de 70 colunas).
pd.set_option('display.max_columns', None)

## 1. Coleta dos dados

Arquivo `listings.csv.gz` (anúncios) do Rio de Janeiro, lido direto da URL.

In [ ]:
# Arquivo listings do RIO DE JANEIRO (insideairbnb.com/get-the-data).
# Pode ser o caminho do arquivo enviado ao Colab ou o link do listings.csv.gz.
URL_LISTINGS = '/content/listings.csv.gz'

In [ ]:
# Colunas usadas na análise (as outras 60+ não são necessárias).
COLUNAS = [
    'id', 'name', 'host_id', 'host_since', 'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'host_identity_verified', 'host_listings_count',
    'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type',
    'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
    'minimum_nights', 'availability_365', 'number_of_reviews', 'first_review', 'last_review',
    'review_scores_rating', 'review_scores_location', 'review_scores_value', 'instant_bookable',
]

## 2. Entendimento inicial

Antes de mexer nos dados: quantas linhas, quais tipos, onde estão os nulos.

In [ ]:
df=pd.read_csv(URL_LISTINGS,usecols=COLUNAS,sep=',')# lendo o arquivo

In [ ]:
df.head() # Conhecendo a base de dados

In [ ]:
df.shape # Linhas x colunas

In [ ]:
# Renomeando as colunas para o português

colunas_traduzidas = {
    'id': 'id',
    'name': 'nome',
    'host_id': 'anfitriao_id',
    'host_since': 'anfitriao_desde',
    'host_response_rate': 'taxa_resposta_anfitriao',
    'host_acceptance_rate': 'taxa_aceitacao_anfitriao',
    'host_is_superhost': 'anfitriao_superhost',
    'host_identity_verified': 'anfitriao_identidade_verificada',
    'host_listings_count': 'total_anuncios_anfitriao',
    'neighbourhood_cleansed': 'bairro',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'property_type': 'tipo_propriedade',
    'room_type': 'tipo_quarto',
    'accommodates': 'acomoda',
    'bathrooms_text': 'banheiros_texto',
    'bedrooms': 'quartos',
    'beds': 'camas',
    'amenities': 'comodidades',
    'price': 'preco',
    'minimum_nights': 'minimo_noites',
    'availability_365': 'disponibilidade_365',
    'number_of_reviews': 'numero_avaliacoes',
    'first_review': 'primeira_avaliacao',
    'last_review': 'ultima_avaliacao',
    'review_scores_rating': 'nota_geral',
    'review_scores_location': 'nota_localizacao',
    'review_scores_value': 'nota_custo_beneficio',
    'instant_bookable': 'reserva_instantanea'
}

df.rename(columns=colunas_traduzidas, inplace=True)
display(df.head())

In [ ]:
df.info()# Conhecendo os tipos de dados

In [ ]:
df.isna().sum().sum()# Identificando quantos valores nulos há

In [ ]:
# Salvando em uma variável os dados nulos
df_nulos=df.isna().stack()

In [ ]:
# Aplicando a máscara, ou seja, filtrando o resultado do outro filtro
df_nulos = df_nulos[df_nulos]


In [ ]:
# Listando a localização dos dados nulos pelo índice
lista_nul = df_nulos.index.to_list()


## 3. Tratamento

Em cada etapa que remove linhas, mostrar quantas linhas existiam antes e depois.

### 3.1 Duplicadas

In [ ]:
df.duplicated().sum() # Nossa base de dados não possui duplicatas, portanto não há o que remover

In [ ]:
# O id é único por anúncio, então também checamos o mesmo imóvel cadastrado mais de uma vez
# (mesmo anfitrião, nome e localização)
df.duplicated(subset=['anfitriao_id', 'nome', 'latitude', 'longitude']).sum()

# Nenhuma duplicata encontrada; nada foi removido.

### 3.2 Preço
O preço vem como texto: `"$1,234.00"`. É preciso remover `$` e `,` e converter para float.

In [ ]:
df['preco']

In [ ]:
df['preco']=df['preco'].str.replace('$','',regex=False).str.replace(',','',regex=False)

In [ ]:
df['preco']=df['preco'].astype(float)

### 3.3 Percentuais
`"95%"` vira `0.95`.

In [ ]:
# As taxas vêm como texto ("95%"): tiramos o '%', convertemos para número e dividimos por 100
colunas_taxa = ['taxa_resposta_anfitriao', 'taxa_aceitacao_anfitriao']

for coluna in colunas_taxa:
    df[coluna] = df[coluna].replace('N/A', np.nan)
    df[coluna] = df[coluna].astype(str).str.replace('%', '', regex=False).astype(float) / 100

In [ ]:
df[colunas_taxa].describe() # Conferindo: os valores devem estar entre 0 e 1

### 3.4 Colunas t/f

In [ ]:
mapeamento_de_cols={

                    't':True,
                    'f':False}

In [ ]:
# Os valores 't' e 'f' viram True e False nas três colunas de sim/não, para facilitar o tratamento e a leitura
colunas_tf = ['anfitriao_superhost', 'anfitriao_identidade_verificada', 'reserva_instantanea']
df[colunas_tf] = df[colunas_tf].replace(mapeamento_de_cols)

In [ ]:
df[colunas_tf].head() # Visualizando a alteração dos dados

### 3.5 Datas

In [ ]:
datas_null=df[['primeira_avaliacao','ultima_avaliacao']]


In [ ]:
df_aval=df[(df['primeira_avaliacao'].isna()) & (df['ultima_avaliacao'].isna())]

In [ ]:
# Anúncios sem nenhuma avaliação não têm notas, então não servem para as análises de avaliação: removemos
print('Linhas antes:', len(df))
df = df.drop(index=df_aval.index)
print('Linhas depois:', len(df))

# Convertendo as colunas de data de texto para datetime
df['anfitriao_desde'] = pd.to_datetime(df['anfitriao_desde'])
df['primeira_avaliacao'] = pd.to_datetime(df['primeira_avaliacao'])
df['ultima_avaliacao'] = pd.to_datetime(df['ultima_avaliacao'])

In [ ]:
df[['anfitriao_desde', 'primeira_avaliacao', 'ultima_avaliacao']].info()  # agora devem ser datetime64

### 3.6 Banheiros
`"1.5 baths"` vira `1.5`.

In [ ]:
# Substituindo 'Half-bath' por '0.5' para que possa ser convertido em número
df['banheiros_texto'] = df['banheiros_texto'].str.replace('(?i)half-bath', '0.5', regex=True)

# Extraindo apenas os números (ponto flutuante ou inteiros) ignorando textos e espaços
df['banheiros_texto'] = df['banheiros_texto'].str.extract(r'(\d+\.?\d*)')[0]

# Convertendo a coluna para tipo numérico correto (float) para aceitar valores decimais (ex: 1.5)
df['banheiros_texto'] = df['banheiros_texto'].astype(float)

# Visualizando o resultado
df['banheiros_texto'].head()

### 3.7 Nulos nas avaliações
Anúncios sem nenhuma avaliação têm `review_scores_*` nulos. *(decisão e justificativa)*

In [ ]:
df

In [ ]:
print('Linhas antes:', len(df))
df = df.dropna(subset=['nota_geral', 'nota_localizacao', 'nota_custo_beneficio'])
print('Linhas depois:', len(df))
# Foi decidido remover as linhas com scores = NaN, pois sem avaliações não há resultados; é desnecessário manter esse tipo de dado na nossa base

## 4. Outliers de preço

In [ ]:
# Tamanho da base antes de remover os outliers
print('Linhas antes:', len(df))

In [ ]:
q1=df['preco'].quantile(0.25)
q3=df['preco'].quantile(0.75)
IQR=q3-q1

limite_superior= q3+1.5*IQR
limite_inferior=q1-1.5*IQR

# Identificando os outliers
outliers= (df['preco']>limite_superior) | (df['preco']<limite_inferior)

In [ ]:
df.drop(index=outliers[outliers].index, inplace=True)
print('Linhas depois:', len(df))

In [ ]:
outliers.sum() # Quantidade de anúncios removidos como outliers

**Decisão sobre os outliers:** os anúncios com preço fora dos limites do IQR foram removidos. Preços extremos distorcem a média e não representam a hospedagem típica, que é o que a análise quer mostrar.

## 5. Transformações

### 5.1 Comodidades
`'["Wifi", "TV", "Kitchen"]'` é uma string, não uma lista.

In [ ]:
# Transformamos em lista as comodidades e usamos o explode pra quebrar essas listas e fazê-las virar linhas
import ast

# Usando ast.literal_eval mapeado na coluna para reconstruir as listas com segurança
df['comodidades'] = df['comodidades'].map(ast.literal_eval)

# Agora aplicamos o explode para expandir as comodidades em novas linhas
df_explodido = df.explode('comodidades')
df_explodido

In [ ]:
df_explodido.reset_index(inplace=True)

In [ ]:
df_explodido.drop(columns='index', inplace=True)

In [ ]:
df_explodido

### 5.2 Ano de entrada do anfitrião
Nova coluna com o ano de `host_since`.

In [ ]:
df['anfitriao_desde'].isna().sum()
# Se algum anfitrião estiver sem data de entrada, usamos a data da primeira avaliação como aproximação

In [ ]:
df['anfitriao_desde'] = df['anfitriao_desde'].fillna(df['primeira_avaliacao'])

### 5.3 Conferência final do tratamento
Tipos e nulos depois de todas as etapas.

In [ ]:
df.info() # Conferindo os tipos e os nulos depois de todas as etapas de tratamento

## 6. Análises

### P1. Os 10 bairros mais caros e os 10 mais baratos
Preço médio por noite (já sem os outliers), considerando só bairros com pelo menos 30 anúncios, para não comparar bairros com poucos casos.

In [ ]:
# Só entram bairros com pelo menos 30 anúncios
anuncios_por_bairro = df['bairro'].value_counts()
bairros_validos = anuncios_por_bairro[anuncios_por_bairro >= 30].index

# Preço médio de cada bairro, do maior para o menor
preco_bairro = df[df['bairro'].isin(bairros_validos)].groupby('bairro')['preco'].mean().sort_values(ascending=False)

# Top 10 bairros mais caros
caros = preco_bairro.head(10)
caros

In [ ]:
# Top 10 bairros mais baratos
baratos = preco_bairro.tail(10).sort_values()
baratos

In [ ]:
# Gráfico dos 10 bairros mais caros
plt.figure(figsize=(10, 5))
sns.barplot(x=caros.values, y=caros.index, hue=caros.index, palette='Reds_r', legend=False)
plt.title('Top 10 bairros mais caros (preço médio por noite)')
plt.xlabel('Preço médio (R$)')
plt.ylabel('Bairro')
plt.show()

In [ ]:
# Gráfico dos 10 bairros mais baratos
plt.figure(figsize=(10, 5))
sns.barplot(x=baratos.values, y=baratos.index, hue=baratos.index, palette='Greens_r', legend=False)
plt.title('Top 10 bairros mais baratos (preço médio por noite)')
plt.xlabel('Preço médio (R$)')
plt.ylabel('Bairro')
plt.show()

**Leitura dos gráficos:** *(completar depois de rodar com os dados do Rio)*

### P2. Preço por tipo de acomodação nos bairros com mais anúncios

In [ ]:
# Os 10 bairros com mais anúncios
top_bairros = df['bairro'].value_counts().head(10).index

# Preço médio de cada tipo de acomodação (linhas: bairros, colunas: tipos)
df[df['bairro'].isin(top_bairros)].pivot_table(index='bairro', columns='tipo_quarto', values='preco', aggfunc='mean').round(2)

### P3. Superhosts cobram mais e têm avaliações melhores?

In [ ]:
# Comparação de preço médio e nota geral média entre Superhosts e não-Superhosts
df.groupby('anfitriao_superhost')[['preco', 'nota_geral']].mean().round(2)

### P4. Comodidades mais comuns e o efeito da piscina no preço

In [ ]:
# As 20 comodidades mais comuns (uma linha por comodidade no df_explodido)
df_explodido['comodidades'].value_counts().head(20)

In [ ]:
# Os nomes vêm com variações ("Pool", "Shared outdoor pool"...), então buscamos 'pool' no texto,
# ignorando falsos positivos: "Pool table" (sinuca), "Pool view" (só vista) e "Whirlpool" (marca de eletrodoméstico).
piscina = df_explodido['comodidades'].str.contains('pool', case=False, na=False) & \
          ~df_explodido['comodidades'].str.contains('table|view|whirlpool', case=False, na=False)

# O df_explodido serve para ACHAR quem tem piscina; a marcação vai no df (1 linha por anúncio)
df['tem_piscina'] = df['id'].isin(df_explodido.loc[piscina, 'id'])

df['tem_piscina'].value_counts()

In [ ]:
# Preço médio de anúncios com e sem piscina
df.groupby('tem_piscina')['preco'].mean().round(2)

### P5. Quantos anfitriões entraram no Airbnb a cada ano?
Considera os anfitriões que têm anúncios ativos na data da extração.

In [ ]:
df['ano'] = df['anfitriao_desde'].dt.strftime('%Y') # Criamos a coluna ano para identificar a quantidade de anfitriões por ano

In [ ]:
# Um anfitrião pode ter vários anúncios: drop_duplicates deixa cada anfitrião uma vez só
anfitrioes_por_ano = df.drop_duplicates(subset='anfitriao_id')['ano'].value_counts().sort_index()

anfitrioes_por_ano.plot(figsize=(10, 5), marker='o', title='Anfitriões que entraram no Airbnb por ano')
plt.xlabel('Ano')
plt.ylabel('Quantidade de anfitriões')
plt.show()

anfitrioes_por_ano

### P6. Diária média em dólar
Cotação do dólar pela API do Banco Central (série 1, dólar comercial).

In [ ]:
URL_DOLAR = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados/ultimos/1?formato=json'

In [ ]:
# Requisição para a API do Banco Central para obter a cotação atual do dólar
response = requests.get(URL_DOLAR)
dados_dolar = response.json()

# Extraindo o valor da taxa de venda do dólar comercial
cotacao_dolar = float(dados_dolar[0]['valor'])

# Calculando a média em dólar
media = df['preco'].mean()
diaria_dol = media / cotacao_dolar

print(f"Cotação do Dólar obtida: R$ {cotacao_dolar:.2f}")
print(f"Diária média em Dólar: $ {diaria_dol:.2f}")

**Leitura:** *(completar depois de rodar com os dados do Rio)*

## 7. Exportação

Salva a base tratada em CSV e em um banco SQLite.

In [ ]:
# Exportando os dados tratados para formato CSV
df.to_csv('listings_rio_tratado.csv', index=False)

In [ ]:
# Criando um banco SQLite e salvando a base tratada na tabela 'listings'
from sqlalchemy import create_engine

engine = create_engine('sqlite:///airbnb_rio.db')

# O SQLite não guarda listas: salvamos as comodidades como texto
df_sql = df.copy()
df_sql['comodidades'] = df_sql['comodidades'].astype(str)
df_sql.to_sql('listings', engine, index=False, if_exists='replace')

In [ ]:
# Consulta SQL: os 5 bairros com maior preço médio
consulta = '''
SELECT bairro, ROUND(AVG(preco), 2) AS preco_medio, COUNT(*) AS anuncios
FROM listings
GROUP BY bairro
ORDER BY preco_medio DESC
LIMIT 5
'''
pd.read_sql(consulta, engine)

## 8. Conclusões

| Pergunta | Resposta |
| --- | --- |
| P1 | |
| P2 | |
| P3 | |
| P4 | |
| P5 | |
| P6 | |

**Limitações:** o preço analisado é o anunciado, não o valor pago pelo hóspede; os dados são de uma única extração; poucos anúncios com piscina deixam a comparação da P4 menos confiável.

**Próximos passos:** *(ex.: modelo de previsão de preço, comparar com outras cidades...)*